<a href="https://colab.research.google.com/github/HRashidLiaquat/lessons-learned/blob/Logstic_Regression_FashionMNIST/Demo1_Project_Reg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

0 Import important libriries

1 Get data Ready

3 Data preprocessing

4 Show Raw data by using matpolab

5 build a Modul

6 Bulid a traning loop

7 define loss and activaction function

8 Module validaction

9 Test with New data point

10 Save Model **bold text**


O Import important lobraries

In [3]:
import torch
import torch.nn as nn
import time
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

**Get Data Ready(turn into tensors, and batches)**

In [5]:
train_dataset = datasets.FashionMNIST(root = '/dataset', train= True, transform = transforms.ToTensor(), download=True)
test_dataset  = datasets.FashionMNIST(root = '/dataset', train = False, transform=transforms.ToTensor(), download=True)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 210kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.93MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.5MB/s]


In [7]:
train_dataset,test_dataset

(Dataset FashionMNIST
     Number of datapoints: 60000
     Root location: /dataset
     Split: Train
     StandardTransform
 Transform: ToTensor(),
 Dataset FashionMNIST
     Number of datapoints: 10000
     Root location: /dataset
     Split: Test
     StandardTransform
 Transform: ToTensor())

**Data Convert into batch**

In [11]:
batch_size = 64

In [15]:
train_loade = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
testloader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

In [16]:
len(train_loade),len(testloader)

(938, 157)

**Bulid a Logstic Regression Model**

In [30]:
class LogsticRegression(nn.Module):
  def __init__(self,input_dim,n_class):
    super().__init__()
    self.linear=nn.Linear(input_dim,n_class)

  def forward(self,x):
    out= self.linear(x)
    return out

In [36]:
model=LogsticRegression(28 * 28 , 10)

In [37]:
model

LogsticRegression(
  (linear): Linear(in_features=784, out_features=10, bias=True)
)

In [54]:
device= 'cuda' if torch.cuda.is_available() else 'cpu'

In [46]:
learning_rate=1e-3

**Define Loos and optimizer**

In [50]:
criterion = nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)

**Build a traning loop**

In [56]:
num_epochs = 50

In [57]:
for epoch in range(num_epochs):
  print('* ' *10)
  print(f'epochn{epoch +1}')
  running_loss=0.0
  running_acc=0.0
  model.train()
  for i , data in enumerate(train_loade, 1):
    img,label=data
    img=img.view(img.size(0), -1)
    img=img.to(device)
    label=label.to(device)
    out=model(img)
    loss=criterion(out,label)
    running_loss+=loss.item()
    _,pred=torch.max(out,1)
    running_acc+=(pred==label).float().mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if i % 300 == 0:
      print(f'[{epoch+1}/ {num_epochs}] loss: {running_loss/i:.6f}, acc: {running_acc/i:.6f}')
  print(f'Finish {epoch +1} epoch,loss : {running_loss/i:.6f}, acc:{running_acc/i:.6f}')

* * * * * * * * * * 
epochn1
[1/ 50] loss: 0.717149, acc: 0.774635
[1/ 50] loss: 0.716969, acc: 0.775078
[1/ 50] loss: 0.714004, acc: 0.776111
Finish 1 epoch,loss : 0.715520, acc:0.775853
* * * * * * * * * * 
epochn2
[2/ 50] loss: 0.705242, acc: 0.776250
[2/ 50] loss: 0.700900, acc: 0.780365
[2/ 50] loss: 0.700846, acc: 0.780087
Finish 2 epoch,loss : 0.701691, acc:0.779817
* * * * * * * * * * 
epochn3
[3/ 50] loss: 0.689224, acc: 0.781927
[3/ 50] loss: 0.690173, acc: 0.784323
[3/ 50] loss: 0.687920, acc: 0.784740
Finish 3 epoch,loss : 0.689327, acc:0.784165
* * * * * * * * * * 
epochn4
[4/ 50] loss: 0.689357, acc: 0.780000
[4/ 50] loss: 0.680231, acc: 0.786380
[4/ 50] loss: 0.678265, acc: 0.787170
Finish 4 epoch,loss : 0.678498, acc:0.786997
* * * * * * * * * * 
epochn5
[5/ 50] loss: 0.672003, acc: 0.787448
[5/ 50] loss: 0.667573, acc: 0.790547
[5/ 50] loss: 0.669134, acc: 0.789809
Finish 5 epoch,loss : 0.668652, acc:0.790095
* * * * * * * * * * 
epochn6
[6/ 50] loss: 0.662845, acc: 0.

**Model validate**

In [60]:
model.eval()
eval_loss=0.
eval_acc=0.
for data in testloader:
  img,label=data
  img=img.view(img.size(0), -1)
  img=img.to(device)
  label=label.to(device)
  with torch.no_grad():
    out=model(img)
    loss=criterion(out,label)
  eval_loss+=loss.item()
  _,pred=torch.max(out,1)
  eval_acc+=(pred==label).float().mean()
print(f'test loss: { eval_loss/len(testloader):.6f}, Acc : {eval_acc / len(train_loade):.6f}')

test loss: 0.546314, Acc : 0.136710
